Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from textblob import TextBlob
import datetime as dt

plt.style.use('ggplot')

Load Dataset

In [2]:
apps_df = pd.read_csv("Play Store Data.csv")
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


Data Cleaning

In [3]:
# Clean Installs
apps_df['Installs'] = apps_df['Installs'].astype(str).str.replace('[+,]', '', regex=True)
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors='coerce')

# Clean Reviews
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')

# Clean Rating
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')

# Clean Size
apps_df['Size_MB'] = apps_df['Size'].astype(str).str.replace('M','', regex=False)
apps_df['Size_MB'] = pd.to_numeric(apps_df['Size_MB'], errors='coerce')

# Clean Date
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')

apps_df.dropna(subset=['Installs','Rating','Reviews','Size_MB','Last Updated'], inplace=True)

Apply Business Filters

In [4]:
filtered_df = apps_df[
    (apps_df['Rating'] >= 4.2) &
    (~apps_df['App'].str.contains(r'\d', regex=True)) &
    (apps_df['Category'].str.startswith(('T','P'))) &
    (apps_df['Reviews'] > 1000) &
    (apps_df['Size_MB'].between(20,80))
]

filtered_df.shape

(139, 14)

Translate Categories

In [5]:
category_map = {
    "Travel & Local": "Voyage et Local",     # French
    "Productivity": "Productividad",         # Spanish
    "Photography": "写真"                    # Japanese
}

filtered_df['Translated_Category'] = filtered_df['Category'].replace(category_map)

C:\Users\Pratham H Shetty\AppData\Local\Temp\ipykernel_21784\3425514119.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Translated_Category'] = filtered_df['Category'].replace(category_map)


Create Time Columns

In [6]:
filtered_df['Month'] = filtered_df['Last Updated'].dt.to_period('M').astype(str)

C:\Users\Pratham H Shetty\AppData\Local\Temp\ipykernel_21784\862447152.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Month'] = filtered_df['Last Updated'].dt.to_period('M').astype(str)


Monthly Aggregation

In [7]:
monthly_df = filtered_df.groupby(['Month','Translated_Category'])['Installs'].sum().reset_index()
monthly_df.head()

,Month,Translated_Category,Installs
0,2014-11,PHOTOGRAPHY,1000000.0
1,2016-10,PRODUCTIVITY,1000000.0
2,2016-12,PERSONALIZATION,2000000.0
3,2017-03,PHOTOGRAPHY,50000000.0
4,2017-06,PHOTOGRAPHY,10000000.0


Calculate Monthly Growth %

In [8]:
monthly_df['Growth_%'] = monthly_df.groupby('Translated_Category')['Installs'].pct_change()*100
monthly_df['Highlight'] = monthly_df['Growth_%'] > 25

Time-Based Visibility (4 PM – 6 PM IST)

In [17]:
current_time = dt.datetime.now().time()
start = dt.time(16,0,0)
end = dt.time(18,0,0)

show_chart = start <= current_time <= end
show_chart

False

Stacked Area Chart (Only If Time Matches)

In [18]:
if show_chart:
    fig = px.area(
        monthly_df,
        x='Month',
        y='Installs',
        color='Translated_Category',
        title='Cumulative Installs Over Time by Category'
    )
    fig.show()
else:
    print("Chart Hidden — Visible only between 4 PM to 6 PM IST")

Chart Hidden — Visible only between 4 PM to 6 PM IST
